# 01 · Setup and connection

Confirms this project can connect to your local ORD database and that the schema looks as expected. If this errors out, see [`docs/local_ord_database.md`](../docs/local_ord_database.md) to build the database first.

In [1]:
from dotenv import load_dotenv
from sqlalchemy import inspect

from ord_analysis.db import get_engine

load_dotenv()
engine = get_engine()

# Deliberately not printing the engine/connection string -- it embeds
# the DB host, which we don't want baked into a committed notebook output.
with engine.connect() as conn:
    print(f"Connected (dialect: {engine.dialect.name})")

Connected (dialect: postgresql)


## Schemas and tables

Every ORD protobuf message type gets its own table under the `ord` schema; RDKit cartridge data (if installed) lives under `rdkit`.

In [2]:
inspector = inspect(engine)
for schema in inspector.get_schema_names():
    tables = inspector.get_table_names(schema=schema)
    if not tables:
        continue
    print(f"schema: {schema} ({len(tables)} tables)")
    for table in sorted(tables):
        print(f"  - {table}")

schema: derived (3 tables)
  - compound_smiles
  - product_compound_smiles
  - reaction_smiles
schema: information_schema (4 tables)
  - sql_features
  - sql_implementation_info
  - sql_parts
  - sql_sizing
schema: ord (63 tables)
  - addition_device
  - addition_speed
  - amount
  - analysis
  - atmosphere
  - compound
  - compound_identifier
  - compound_preparation
  - crude_component
  - current
  - data
  - dataset
  - date_time
  - electrochemistry_cell
  - electrochemistry_conditions
  - electrochemistry_measurement
  - float_value
  - flow_conditions
  - flow_rate
  - illumination_conditions
  - length
  - mass
  - mass_spec_measurement_details
  - moles
  - percentage
  - person
  - pressure
  - pressure_conditions
  - pressure_control
  - pressure_measurement
  - product_compound
  - product_measurement
  - reaction
  - reaction_conditions
  - reaction_environment
  - reaction_identifier
  - reaction_input
  - reaction_notes
  - reaction_observation
  - reaction_outcome
  - r

## Row counts for a few key tables

In [3]:
from sqlalchemy import func, select
from sqlalchemy.orm import Session

from ord_schema.orm.mappers import Mappers

with Session(engine) as session:
    for name in ["Dataset", "Reaction", "Compound", "ProductCompound"]:
        count = session.execute(select(func.count()).select_from(Mappers[name])).scalar_one()
        print(f"{name}: {count} rows")

Dataset: 53 rows
Reaction: 2428291 rows
Compound: 17043157 rows
ProductCompound: 2673037 rows
